# Create a hold-out set for final testing

Author: Pete King

This notebook creates a separate testing dataset to hold in reserve for a final evaluation of our selected model's ability to generalize to unseen data.

In [1]:
#123456789012345678901234567890123456789012345678901234567890123456789012345678
import re

import pandas as pd

import data_prep as dp

In [2]:
# Select the data file and start date for the testing set
# ----------------------------------------------------------------------------
# We want to rigorously test our final model against a period of dynamic
# real-world volatility.  For example, beginning in October 2024 would capture
# an initial period of relatively low volatility, followed by the highly 
# volatility period after Trump announced reciprocal tarrifs in April 2025.
# ----------------------------------------------------------------------------
DATA_FILENAME = 'raw_data_prediction_dataset.csv'
TESTING_START_DATE = '2024-10-01'
# Select buffer size between validation/test and training sets
# We calculate target variables using 63 days of "future" historical data
# A buffer of one quarter (63 business days) guards against data leakage
BUFFER_SIZE = 63

In [3]:
# Import feature dataset with labels
df = pd.read_csv(DATA_FILENAME)
df

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,TLT_logreturn_target,TLT_volatility_target,LQD_logreturn_target,LQD_volatility_target,HYG_logreturn_target,HYG_volatility_target,TIP_logreturn_target,TIP_volatility_target,GLD_logreturn_target,GLD_volatility_target
0,1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25021,2026-03-30,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25022,2026-03-31,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25023,2026-04-01,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25024,2026-04-02,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Let's look at the earliest starting dates for each of our column labels to see how far back our labels go for each ETF.

In [4]:
label_tag = '_volatility_target'
# Use a regular expression to parse the ticker symbol from the column name
p = re.compile(label_tag)
start_dates = {}
for column_name in df.columns:
    if label_tag in column_name:
        ticker = p.split(column_name)[0]
        start_dates[ticker] = (
            df[['date', column_name]].dropna().date.iloc[0]
        )
sorted([(k, v) for k, v in start_dates.items()], key=lambda pair: pair[1])

[('XLF', '1998-12-22'),
 ('XLK', '1998-12-22'),
 ('XLU', '1998-12-22'),
 ('XLV', '1998-12-22'),
 ('XLE', '1998-12-22'),
 ('XLI', '1998-12-22'),
 ('XLB', '1998-12-22'),
 ('XLP', '1998-12-22'),
 ('XLY', '1998-12-22'),
 ('IEF', '2002-07-30'),
 ('TLT', '2002-07-30'),
 ('LQD', '2002-07-30'),
 ('TIP', '2003-12-05'),
 ('GLD', '2004-11-18'),
 ('HYG', '2007-04-11'),
 ('BIL', '2007-05-30'),
 ('XLRE', '2015-10-08')]

In [5]:
print(f'The XLRE (Real Estate) series only goes back to {start_dates['XLRE']}')

The XLRE (Real Estate) series only goes back to 2015-10-08


For the Vector AutoRegressive Integrated Moving Average (VARIMA) and the multivariate Long Short-Term Memory (LSTM) models, we intend to use fluctuations in the price of all labeled ETF price time series as features.  However, we've observed that a few of these time series have significantly later start dates than the others.  For example, if we drop the 'XLRE' ETF (Real Estate) from the dataset, we'll free up 8 years worth of training data for the rest of the ETFs.

Going forward, we plan to test three variations of the data:

 - Including all raw data from roughly 2015 (start of XLRE data)
 - Dropping XLRE (Real Estate) to make data as early as 2007 available
 - Dropping HYG and BIL to make data as early as 2004 available

In [6]:
train_val_df, test_df = dp.time_series_split(
    df, TESTING_START_DATE, BUFFER_SIZE
)
train_val_df

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,TLT_logreturn_target,TLT_volatility_target,LQD_logreturn_target,LQD_volatility_target,HYG_logreturn_target,HYG_volatility_target,TIP_logreturn_target,TIP_volatility_target,GLD_logreturn_target,GLD_volatility_target
0,1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24569,2024-06-27,29147.044,23286.508,119.50314,1103.565,313.044,318.39,123.539,122.677,144.869,...,0.228628,0.143676,0.224334,0.062757,0.204915,0.037339,0.140210,0.040556,0.554863,0.144959
24570,2024-06-28,29147.044,23286.508,119.50314,1103.565,313.044,318.39,123.539,122.677,144.869,...,0.324501,0.138220,0.265942,0.061262,0.224723,0.036886,0.162335,0.040043,0.522621,0.146326
24571,2024-07-01,29511.664,23478.570,120.17172,1146.182,313.569,318.94,123.736,122.911,144.922,...,0.375021,0.133504,0.279868,0.060026,0.222517,0.036968,0.175744,0.039015,0.480090,0.147709
24572,2024-07-02,29511.664,23478.570,120.17172,1146.182,313.569,318.94,123.736,122.911,144.922,...,0.372776,0.133401,0.273494,0.059539,0.209581,0.036876,0.181298,0.039285,0.522023,0.148631


In [7]:
test_df

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,TLT_logreturn_target,TLT_volatility_target,LQD_logreturn_target,LQD_volatility_target,HYG_logreturn_target,HYG_volatility_target,TIP_logreturn_target,TIP_volatility_target,GLD_logreturn_target,GLD_volatility_target
24637,2024-10-01,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,-0.438186,0.145855,-0.185906,0.075138,-0.002814,0.041414,-0.136964,0.041522,-0.057080,0.161541
24638,2024-10-02,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,-0.393358,0.145477,-0.176681,0.075096,0.006337,0.041696,-0.127348,0.041715,-0.003910,0.163856
24639,2024-10-03,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,-0.368308,0.144630,-0.160094,0.074250,0.021946,0.041511,-0.118571,0.041140,-0.032721,0.164609
24640,2024-10-04,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,-0.336710,0.143062,-0.148419,0.073708,0.033552,0.041571,-0.091239,0.039062,-0.029661,0.164580
24641,2024-10-07,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,-0.351820,0.143927,-0.146398,0.073588,0.037505,0.041190,-0.093240,0.039113,0.006384,0.164838
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25021,2026-03-30,31442.483,24065.956,122.49035,1227.495,327.460,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25022,2026-03-31,31442.483,24065.956,122.49035,1227.495,327.460,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25023,2026-04-01,31442.483,24065.956,122.49035,1227.495,327.460,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25024,2026-04-02,31442.483,24065.956,122.49035,1227.495,327.460,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The gap between the training and testing sets guards against data leakage.

In [8]:
# Save segmented datasets for further analysis; drop the index
train_val_df.to_csv('train_val_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)